# SEED2S + Reveal — South America Critical Minerals Analysis
**Region:** Latin America and the Caribbean (South America sub-region)  
**Data Source:** U.S. Census Bureau / USITC — Annual customs import values (2015–2025)

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

---
## 2. Load and Combine Data

Each CSV has 2 metadata header rows (title + export date) before the actual column headers. We skip those and combine all 4 files into one DataFrame.

In [ ]:
csv_files = [
    'Copper & Graphite.csv',
    'Lithium, Cobalt, Sliver.csv',
    'Nickel, Manganese, Niobium.csv',
    'Tin, Zinc.csv'
]

frames = []
for f in csv_files:
    df_temp = pd.read_csv(f, skiprows=2)
    frames.append(df_temp)

df = pd.concat(frames, ignore_index=True)

# Drop the trailing empty column produced by the trailing comma in the CSV
df = df.loc[:, ~df.columns.str.fullmatch('Unnamed.*')]

print(f'Combined shape: {df.shape}')
df.head()

---
## 3. Rename and Parse Columns

- Extract the 4-digit **HS Code** and the **Commodity Name** from the `Commodity` column.
- Rename `Time` → `Year` and `Customs Value (Gen) ($US)` → `Nominal Value`.

> **Note:** The raw data is annual — there is no monthly breakdown, so `Month` will be set to `NaN`. Month-level analyses in the framework will be noted as not applicable for this dataset.

In [ ]:
# Extract HS Code (first 4 characters) and Commodity Name (remainder)
df['HS Code'] = df['Commodity'].str[:4].str.strip()
df['Commodity Name'] = df['Commodity'].str[4:].str.strip()

# Rename columns
df.rename(columns={
    'Time': 'Year',
    'Customs Value (Gen) ($US)': 'Nominal Value'
}, inplace=True)

# Add Month column (N/A — data is annual only)
df['Month'] = np.nan

# Reorder columns to match framework
df = df[['HS Code', 'Commodity Name', 'Year', 'Month', 'Country', 'Nominal Value']]

print(df.dtypes)
df.head(10)

---
## 4. Filter Countries

The raw data contains aggregate region rows (`South/Central America`, `South America`) alongside individual country rows. We keep only **individual country** rows for analysis to avoid double-counting.

In [ ]:
aggregate_labels = ['South/Central America', 'South America']
df = df[~df['Country'].isin(aggregate_labels)].reset_index(drop=True)

print(f'Rows after removing aggregates: {len(df)}')
print('Countries in dataset:', sorted(df['Country'].unique()))

---
## 5. Filter Year Range (2015–2025)

In [ ]:
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df = df[(df['Year'] >= 2015) & (df['Year'] <= 2025)].reset_index(drop=True)

print(f'Year range in data: {int(df["Year"].min())} – {int(df["Year"].max())}')
print(f'Rows: {len(df)}')

---
## 6. Clean Nominal Value

Values are stored as strings with commas (e.g., `"9,941,921"`). Strip commas and convert to float.

In [ ]:
df['Nominal Value'] = (
    df['Nominal Value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['Nominal Value'] = pd.to_numeric(df['Nominal Value'], errors='coerce')

print(df['Nominal Value'].describe())

---
## 7. Add Annual Inflation Rate and Calculate CPI

U.S. annual CPI inflation rates (source: U.S. Bureau of Labor Statistics).  
**Base year: 2010, Base CPI = 100**  
Formula: `CPI_current = CPI_past × (inflation_rate / 100 + 1)`

In [ ]:
# U.S. annual inflation rates (%)
inflation_rates = {
    2010: 1.64,
    2011: 3.16,
    2012: 2.07,
    2013: 1.46,
    2014: 1.62,
    2015: 0.12,
    2016: 1.26,
    2017: 2.13,
    2018: 2.44,
    2019: 1.81,
    2020: 1.23,
    2021: 4.70,
    2022: 8.00,
    2023: 4.12,
    2024: 2.90,
    2025: 2.80   # estimate
}

# Calculate CPI for each year starting from base CPI = 100 in 2010
cpi_values = {}
cpi = 100.0
for year in range(2010, 2026):
    cpi_values[year] = round(cpi, 4)
    if year < 2025:
        cpi = cpi * (inflation_rates[year + 1] / 100 + 1)
    else:
        cpi = cpi * (inflation_rates[2025] / 100 + 1)

# Fix: recalculate properly — CPI for year Y uses inflation rate OF year Y applied to prior year's CPI
cpi_values = {}
cpi = 100.0
cpi_values[2010] = round(cpi, 4)
for year in range(2011, 2026):
    cpi = cpi * (inflation_rates[year] / 100 + 1)
    cpi_values[year] = round(cpi, 4)

print('CPI by year (Base 2010 = 100):')
for y, c in cpi_values.items():
    print(f'  {y}: {c}')

# Map to DataFrame
df['Annual Inflation Rate'] = df['Year'].map(inflation_rates)
df['CPI'] = df['Year'].map(cpi_values)

---
## 8. Calculate Inflation-Adjusted Real Value

Formula: `Real Value = (Nominal Value / CPI) × 100`

In [ ]:
df['Real Value'] = (df['Nominal Value'] / df['CPI']) * 100

print(df[['HS Code', 'Commodity Name', 'Year', 'Country', 'Nominal Value', 'CPI', 'Real Value']].head(10))

---
## 9. Apply Category Type Classification

Based on the Southeast Asia team's classification framework, commodities are categorized by HS code chapter:

| Category | HS Chapter / Code Range | Description |
|---|---|---|
| Ore / Raw | 25xx, 26xx | Mineral ores, earths, concentrates |
| Compound | 28xx, 7401–7402, 7501–7502, 7901–7902, 8001–8002 | Intermediate compounds, mattes, unrefined metals |
| Refined / Articles | 7403–7407, 7409–7419, 7503–7508, 7903–7907, 8003–8007 | Refined metals and simple articles |
| Advanced Product | 7408 (wire), 7419 (advanced articles), and other manufactured codes | Manufactured / value-added products |

In [ ]:
def classify_hs_code(hs_code):
    """Classify HS code into category type per SEED2S+Reveal methodology."""
    try:
        code = int(hs_code)
    except (ValueError, TypeError):
        return 'Unknown'

    # Ore / Raw — mineral ores and concentrates (Chapters 25–26)
    if 2500 <= code <= 2699:
        return 'Ore / Raw'

    # Compound — intermediate / unrefined products
    if code in [2801, 2802, 2804, 2814, 2819, 2822, 2836]:  # chemical compounds
        return 'Compound'
    if 7401 <= code <= 7402:   # copper mattes, unrefined copper
        return 'Compound'
    if 7501 <= code <= 7502:   # nickel mattes, unrefined nickel
        return 'Compound'
    if 7901 <= code <= 7902:   # unwrought zinc
        return 'Compound'
    if 8001 <= code <= 8002:   # unwrought tin
        return 'Compound'

    # Advanced Product — wire, rods for further manufacture, complex articles
    if code == 7408:           # copper wire
        return 'Advanced Product'
    if code == 7505:           # nickel wire/rods
        return 'Advanced Product'
    if code == 7904:           # zinc wire/rods
        return 'Advanced Product'
    if code == 8003:           # tin bars/rods/wire
        return 'Advanced Product'

    # Refined / Articles — refined metals and simple metal articles
    if 7403 <= code <= 7419:
        return 'Refined / Articles'
    if 7503 <= code <= 7508:
        return 'Refined / Articles'
    if 7903 <= code <= 7907:
        return 'Refined / Articles'
    if 8003 <= code <= 8007:
        return 'Refined / Articles'
    if 8112 <= code <= 8113:   # niobium, germanium
        return 'Refined / Articles'

    return 'Unknown'


df['Category Type'] = df['HS Code'].apply(classify_hs_code)

print('Category distribution:')
print(df['Category Type'].value_counts())

---
## 10. Final Column Selection and Reorder

In [ ]:
df = df[[
    'HS Code',
    'Commodity Name',
    'Year',
    'Month',
    'Country',
    'Nominal Value',
    'Annual Inflation Rate',
    'CPI',
    'Real Value',
    'Category Type'
]]

print(f'Final shape: {df.shape}')
df.head()

---
## 11. Print Missing Values Per Column

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Note: Month is NaN for all rows — raw data is annual only (no monthly breakdown).')

---
## 12. Convert Columns to Numeric Types

In [ ]:
numeric_cols = ['Year', 'Month', 'Real Value', 'Nominal Value', 'Annual Inflation Rate', 'CPI']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Year'] = df['Year'].astype('Int64')

print(df.dtypes)

---
## 13. Check for Negative Values

Checking `Year`, `Month`, `Real Value`, and `Nominal Value` for any negative entries.

In [ ]:
check_cols = ['Year', 'Month', 'Real Value', 'Nominal Value']
for col in check_cols:
    neg_count = (df[col] < 0).sum()
    print(f'{col}: {neg_count} negative value(s)')

---
## 14. Remove Duplicate Rows

Remove rows that are identical across **all** columns.

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)
print(f'Rows before: {before} | Rows after: {after} | Duplicates removed: {before - after}')

---
## 15. Check for Duplicate Dates per HS Code and Country

For each (HS Code, Country) pair, verify there are no repeated Year entries.

In [ ]:
dupes = df[df.duplicated(subset=['HS Code', 'Country', 'Year'], keep=False)]

if dupes.empty:
    print('No duplicate (HS Code, Country, Year) combinations found.')
else:
    print(f'WARNING: {len(dupes)} rows have duplicate (HS Code, Country, Year):')
    print(dupes[['HS Code', 'Commodity Name', 'Country', 'Year', 'Nominal Value']].sort_values(
        ['HS Code', 'Country', 'Year']
    ))

---
## 16. Outlier Detection

Using the IQR method to flag rows where `Real Value` is unusually high or low. **Outliers are not removed** — they are noted for awareness during analysis.

In [ ]:
Q1 = df['Real Value'].quantile(0.25)
Q3 = df['Real Value'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers = df[(df['Real Value'] < lower_fence) | (df['Real Value'] > upper_fence)].copy()

print(f'IQR Fences: [{lower_fence:,.0f}, {upper_fence:,.0f}]')
print(f'Outlier rows flagged: {len(outliers)}')
print()
print('Top outliers (highest Real Value):')
print(
    outliers
    .sort_values('Real Value', ascending=False)
    [['HS Code', 'Commodity Name', 'Country', 'Year', 'Nominal Value', 'Real Value']]
    .head(15)
    .to_string(index=False)
)

---
## 17. Final Cleaned Dataset Summary

In [ ]:
print('=== CLEANED DATASET SUMMARY ===')
print(f'Shape         : {df.shape}')
print(f'Years covered : {int(df["Year"].min())} – {int(df["Year"].max())}')
print(f'Countries     : {df["Country"].nunique()} — {sorted(df["Country"].unique())}')
print(f'HS Codes      : {df["HS Code"].nunique()} unique codes')
print(f'Commodities   : {df["Commodity Name"].nunique()} unique names')
print()
print('Category Type distribution:')
print(df['Category Type'].value_counts())
print()
print('Missing values:')
print(df.isnull().sum())
print()
df.head(10)

---
## 18. Export Cleaned Data to CSV

In [ ]:
df.to_csv('south_america_cleaned.csv', index=False)
print('Saved: south_america_cleaned.csv')